# English Noun Extraction from TripAdvisor Reviews

This notebook loads review data, performs sentiment analysis, extracts noun keywords, calculates word frequencies, and exports the results.

In [ ]:
import nltk
import nltk.corpus
import pandas as pd
import os
import glob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from nltk import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.probability import FreqDist
from nltk.tag import pos_tag
import collections
import operator
import re
import numpy as np

nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download("punkt")
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')

## 1. Data Loading and Transformation

In [ ]:
def read_and_transform_data(file_name):
    '''
    Transform data before EDA
    '''
    # Read data
    data = pd.read_excel(file_name)
    
    # Drop previous index
    data = data.drop(['Unnamed: 0'], axis=1)
    
    # Change string to datetime
    data['write_date'] = pd.to_datetime(data.write_date)
    data['date_dep'] = pd.to_datetime(data.date_dep)
    
    # Filter after 2014 only
    data = data[data['write_date'] >= '2014-1-1']
    
    data['attraction'] = [file.split('.')[-2] for i in range(len(data))]
    
    return data

## 2. Load and Combine Review Files

In [ ]:
path = './Pattaya'
files = glob.glob(path + '/*.xlsx')
df = pd.DataFrame()
for file in files:
    try:
        df_file = read_and_transform_data(file)
        df = pd.concat([df, df_file])
        
    except:
        continue

## 3. Data Preview and Inspection

In [ ]:
# Preview dataset
df.head(3)

In [ ]:
# Preview dataset
df.tail(3)

In [ ]:
# Check data type and null values
df.info()

## 4. Sentiment Analysis Using VADER

In [ ]:
sid_obj = SentimentIntensityAnalyzer()

compound_score = []
sentiment_list = []

for sentence in df['text_review']:
    
    sentiment_dict = sid_obj.polarity_scores(sentence)
    
    if sentiment_dict['compound'] >= 0.05 :
        result = "Positive"
 
    elif sentiment_dict['compound'] <= - 0.05 :
        result = "Negative"
 
    else :
        result = "Neutral"

    compound_score.append(sentiment_dict['compound'])
    sentiment_list.append(result)

### 4.1 Add Sentiment and Time-Based Features

In [ ]:
df['compound_score'] = compound_score
df['sentiment'] = sentiment_list
df['year'] = df['write_date'].dt.year
df['month_year'] = df['write_date'].dt.to_period('M')
df['month'] = df['write_date'].dt.month

In [ ]:
df.head(3)

In [ ]:
df.info()

### 4.2 Sentiment Summary by Year and Month

In [ ]:
summary_year = pd.pivot_table(df, index='year', columns='sentiment', aggfunc='count')
summary_year = summary_year['text_review'].reindex(columns=['Positive', 'Neutral', 'Negative'])

summary_month_year = pd.pivot_table(df, index='month_year', columns='sentiment', aggfunc='count')
summary_month_year = summary_month_year['text_review'].reindex(columns=['Positive', 'Neutral', 'Negative'])

### 4.3 Optional Summary Export

In [ ]:
# suffix = 'summary_year'
# summary_year.to_excel(f'{path.split("/")[-1]}_{suffix}.xlsx')

In [ ]:
# suffix = 'summary_month_year'
# summary_month_year.to_excel(f'{path.split("/")[-1]}_{suffix}.xlsx')

## 5. Text Preprocessing Setup

In [ ]:
lemmatizer = WordNetLemmatizer()
stopwords = set(stopwords.words('english'))

## 6. Year and Sentiment Configuration

In [ ]:
year_list = [i for i in range(2014, 2024)]
sentiments = ['Positive', 'Neutral', 'Negative']

## 7. Noun Extraction and Word-Frequency Analysis

In [ ]:
word_freq_all = {}

for year in year_list:
    
    for sentiment in sentiments:
        
        df_year = df[df['year'] == year]
        df_year = df_year[df_year['sentiment'] == sentiment]
        
        text_clean_list = []

        for review in df_year['text_review']:
            
            text = re.sub(r'\d+', '', review)
            text_token = word_tokenize(text.lower())   
            list_of_words = [word for word in text_token if word.isalnum()]
            list_of_words = [x for x in list_of_words if x not in stopwords]
            list_of_words = [lemmatizer.lemmatize(word) for word in list_of_words]
            list_of_words_tag = pos_tag(list_of_words)
            list_of_words_tag_nouns = [word for word, pos in list_of_words_tag if pos in ['NN', 'NNS', 'NNP', 'NNPS']]

            
            for word in list_of_words_tag_nouns:
                text_clean_list.append(word)
                
        frequency = dict(collections.Counter(text_clean_list))
        sorted_dict = dict(sorted(frequency.items(), key=operator.itemgetter(1), reverse=True))
        sorted_dict_20_keys = list(sorted_dict.keys())[:20]
        sorted_dict_20_values = list(sorted_dict.values())[:20]
        
        word_freq_all[f'{year}_{sentiment}'] = {}
        word_freq_all[f'{year}_{sentiment}']['keywords'] = sorted_dict_20_keys
        word_freq_all[f'{year}_{sentiment}']['word_freq'] = sorted_dict_20_values

### 7.1 Overall Word Frequency by Sentiment

In [ ]:
for sentiment in sentiments:
        
    df_all = df[df['sentiment'] == sentiment]

    text_clean_list = []

    for review in df_all['text_review']:

        text = re.sub(r'\d+', '', review)
        text_token = word_tokenize(text.lower())   
        list_of_words = [word for word in text_token if word.isalnum()]
        list_of_words = [x for x in list_of_words if x not in stopwords]
        list_of_words = [lemmatizer.lemmatize(word) for word in list_of_words]
        list_of_words_tag = pos_tag(list_of_words)
        list_of_words_tag_nouns = [word for word, pos in list_of_words_tag if pos in ['NN', 'NNS', 'NNP', 'NNPS']]

        for word in list_of_words_tag_nouns:
            text_clean_list.append(word)

    frequency = dict(collections.Counter(text_clean_list))
    sorted_dict = dict(sorted(frequency.items(), key=operator.itemgetter(1), reverse=True))
    sorted_dict_20_keys = list(sorted_dict.keys())[:20]
    sorted_dict_20_values = list(sorted_dict.values())[:20]

    word_freq_all[f'all_{sentiment}'] = {}
    word_freq_all[f'all_{sentiment}']['keywords'] = sorted_dict_20_keys
    word_freq_all[f'all_{sentiment}']['word_freq'] = sorted_dict_20_values

## 8. Extract Nouns from All Reviews

In [ ]:
all_keywords = []

for review in df['text_review']:
            
    text = re.sub(r'\d+', '', review)
    text_token = word_tokenize(text.lower())   
    list_of_words = [word for word in text_token if word.isalnum()]
    list_of_words = [x for x in list_of_words if x not in stopwords]
    list_of_words = [lemmatizer.lemmatize(word) for word in list_of_words]
    list_of_words_tag = pos_tag(list_of_words)
    list_of_words_tag_nouns = [word for word, pos in list_of_words_tag if pos in ['NN', 'NNS', 'NNP', 'NNPS']]
    
    all_keywords.append(list_of_words_tag_nouns)

df['all_keywords'] = all_keywords

In [ ]:
df.tail(3)

## 9. Count Reviews Containing Each Keyword

In [ ]:
for year in year_list:
    
    for sentiment in sentiments:
        
        df_year = df[df['year'] == year]
        df_year = df_year[df_year['sentiment'] == sentiment]
        
        keywords = word_freq_all[f'{year}_{sentiment}']['keywords']
        
        token_clean_list = df_year['all_keywords'].to_list()
        review_count_list = []
        for keyword in keywords:
            count_review = 0
            for token_clean in token_clean_list:
                if keyword in token_clean:
                    count_review += 1
                else:
                    pass
            review_count_list.append(count_review)
            
        word_freq_all[f'{year}_{sentiment}']['review_count'] = review_count_list

### 9.1 Overall Review Count by Sentiment

In [ ]:
for sentiment in sentiments:

    df_year = df
    df_year = df_year[df_year['sentiment'] == sentiment]

    keywords = word_freq_all[f'all_{sentiment}']['keywords']
    
    token_clean_list = df_year['all_keywords'].to_list()
    review_count_list = []
    for keyword in keywords:
        count_review = 0
        for token_clean in token_clean_list:
            if keyword in token_clean:
                count_review += 1
            else:
                pass
        review_count_list.append(count_review)

    word_freq_all[f'all_{sentiment}']['review_count'] = review_count_list

## 10. Combine Extracted Keyword Results

In [ ]:
df_extract_result = pd.DataFrame()

for year, data in word_freq_all.items():
    
    year_tag = [str(year).split('_')[0] for i in range(len(data['keywords']))]
    sentiment_tag = [str(year).split('_')[-1] for i in range(len(data['keywords']))]
    keywords = data['keywords']
    word_freq = data['word_freq']
    review_count = data['review_count']

#     year_tag = add_nan_to_list(year_tag)
#     sentiment_tag = add_nan_to_list(sentiment_tag)
#     keywords = add_nan_to_list(keywords)
#     word_freq = add_nan_to_list(word_freq)
#     review_count = add_nan_to_list(review_count)
    
    df_temp = pd.DataFrame({
            'year_tag': year_tag,
            'sentiment_tag': sentiment_tag,
            'keywords': keywords,
            'word_freq': word_freq,
            'review_count': review_count
        })
    
    df_extract_result = pd.concat([df_extract_result, df_temp])

## 11. Export Results

In [ ]:
suffix = 'sentiment_result'
df.to_excel(f'{path.split("/")[-1]}_{suffix}.xlsx')

In [ ]:
suffix = 'word_freq_extract'
df_extract_result.to_excel(f'{path.split("/")[-1]}_{suffix}.xlsx')